In [0]:
from pyspark.sql.functions import col, monotonically_increasing_id, current_timestamp
from delta.tables import DeltaTable

print("CREATING DIM_AIRPORT FROM SILVER LAYER")

silver_airport_path = "s3://travel-analytics-bronze/delta/silver/airports/"
gold_dim_airport_path = "s3://travel-analytics-bronze/delta/gold/Dim_Airport/"

# =============================================================
# STEP 1: LOAD SILVER DATA (Already Transformed)
# =============================================================
print("\nSTEP 1: Loading Silver Airports Data...")

silver_df = spark.read.format("delta").load(silver_airport_path)

print(f"Loaded {silver_df.count():,} airport records from silver")

# =============================================================
# STEP 2: ADD DERIVED COLUMNS & SCD TYPE 2 METADATA
# =============================================================
print("\nSTEP 2: Creating Dimension Structure")
dim_airport_df = (
    silver_df
    
    # Add surrogate key
    .withColumn("Dim_Airport_SK", monotonically_increasing_id() + 1)

    # Select & rename columns to match dimension model
    .select(
        col("Dim_Airport_SK"),                       # PK
        col("Airport_Id").alias("Airport_ID_BK"),    # BK
        col("Airport_Name"),
        col("City"),
        col("Longitude"),
        col("Latitude"),
        col("Airport_Type"),
        current_timestamp().alias("Updated_At")
    )
)

CREATING DIM_AIRPORT FROM SILVER LAYER

STEP 1: Loading Silver Airports Data...
Loaded 105 airport records from silver

STEP 2: Creating Dimension Structure


In [0]:
dim_airport_df.display()

Dim_Airport_SK,Airport_ID_BK,Airport_Name,City,Longitude,Latitude,Airport_Type,Updated_At
1,94,Ignatyevo Airport,Blagoveschensk,127.4125,50.4254,DOMESTIC,2025-12-13T04:02:13.681Z
2,29,Ust-Ilimsk Airport,Ust Ilimsk,102.5658,58.1361,DOMESTIC,2025-12-13T04:02:13.681Z
3,88,Tolmachevo Airport,Novosibirsk,82.6507,55.0126,DOMESTIC,2025-12-13T04:02:13.681Z
4,56,Sovetskiy Airport,Sovetskiy,63.6019,61.3266,DOMESTIC,2025-12-13T04:02:13.681Z
5,24,Kurumoch International Airport,Samara,50.1643,53.5049,INTERNATIONAL,2025-12-13T04:02:13.681Z
6,99,Ulan-Ude Airport (Mukhino),Ulan-ude,107.438,51.8078,DOMESTIC,2025-12-13T04:02:13.681Z
7,39,Kogalym International Airport,Kogalym,74.5338,62.1904,INTERNATIONAL,2025-12-13T04:02:13.681Z
8,102,Barnaul Airport,Barnaul,83.5385,53.3638,DOMESTIC,2025-12-13T04:02:13.681Z
9,38,Pskov Airport,Pskov,28.3956,57.7839,DOMESTIC,2025-12-13T04:02:13.681Z
10,15,Mineralnyye Vody Airport,Mineralnye Vody,43.0819,44.2251,DOMESTIC,2025-12-13T04:02:13.681Z


In [0]:
# =============================================================
# STEP 3: SAVE TO GOLD LAYER
# =============================================================
print("\nSTEP 3: Saving to Gold Layer...")
dim_airport_df.write \
    .format("delta") \
    .mode("overwrite") \
    .save(gold_dim_airport_path)

print(f"Successfully created Dim_Airport with {dim_airport_df.count():,} records")


STEP 3: Saving to Gold Layer...
Successfully created Dim_Airport with 105 records
